## 1. Configuration and Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    current_timestamp,
    desc,
    from_json,
    max,
    min,
    sum,
    to_timestamp,
    unix_timestamp,
    when,
)
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
import os

## 2. Create Spark Session with Delta Lake + Kafka

In [ ]:
# JARs configuration (Delta + Kafka)
jar_path = os.path.abspath("jars/delta-spark_2.12-3.2.1.jar") + "," + \
           os.path.abspath("jars/delta-storage-3.2.1.jar")

spark = SparkSession.builder \
    .appName("Silver_Pipeline_Kafka_to_Delta") \
    .config("spark.jars", jar_path) \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .getOrCreate()  # type: ignore[attr-defined]

spark.sparkContext.setLogLevel("WARN")
print(f"✓ Spark Session created (version {spark.version})")

your 131072x1 screen size is bogus. expect trouble
25/12/16 12:15:58 WARN Utils: Your hostname, PCFlo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/16 12:15:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/12/16 12:15:58 WARN Utils: Your hostname, PCFlo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/16 12:15:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/flori/OneDrive/Documents/dev/simple-streaming-pipeline/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/fabgrall/.ivy2/cache
The jars for the packages stored in: /home/fabgrall/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2141baae-172c-470d-b625-e2524e2ec1c3;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.

✓ Spark Session créée (version 3.5.3)


## 3. Kafka and Path Configuration

In [ ]:
# Kafka configuration
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"
KAFKA_TOPIC = "iot_sensors"

# Delta Lake paths
OUTPUT_PATH = "output/delta/silver/sensor_data"
CHECKPOINT_PATH = "checkpoints/silver"

print(f"📡 Kafka: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"📡 Topic: {KAFKA_TOPIC}")
print(f"📁 Output Path: {OUTPUT_PATH}")
print(f"📁 Checkpoint Path: {CHECKPOINT_PATH}")

📡 Kafka: localhost:9092
📡 Topic: iot_sensors
📁 Output Path: output/delta/silver/sensor_data
📁 Checkpoint Path: checkpoints/silver


## 4. Kafka Message Schema

IoT message JSON structure:
- `timestamp`: ISO 8601 timestamp
- `device_id`: Sensor identifier
- `building`: Building
- `floor`: Floor number
- `type`: Measurement type
- `value`: Measured value
- `unit`: Unit of measurement

In [ ]:
sensor_schema = StructType([
    StructField("timestamp", StringType(), False),
    StructField("device_id", StringType(), False),
    StructField("building", StringType(), False),
    StructField("floor", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

print("✓ Schema defined")

✓ Schéma défini


## 5. Read Kafka Stream

Kafka message consumption:
- JSON deserialization from `value`
- Kafka metadata extraction (partition, offset, timestamp)

In [ ]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✓ Kafka stream configured")
print(f"Kafka schema: {kafka_stream.schema}")

✓ Stream Kafka configuré
Schéma Kafka: StructType([StructField('key', BinaryType(), True), StructField('value', BinaryType(), True), StructField('topic', StringType(), True), StructField('partition', IntegerType(), True), StructField('offset', LongType(), True), StructField('timestamp', TimestampType(), True), StructField('timestampType', IntegerType(), True)])


In [ ]:
# Deserialize JSON and extract metadata
parsed_stream = kafka_stream \
    .select(
        from_json(col("value").cast("string"), sensor_schema).alias("data"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("timestamp").alias("kafka_timestamp")
    ) \
    .select("data.*", "kafka_partition", "kafka_offset", "kafka_timestamp")

print("✓ JSON deserialized")

✓ JSON désérialisé


## 6. Silver Transformations - Enrichments

### 6.1 Comfort Index
Temperature classification:
- **Cold**: < 18°C
- **Comfortable**: 18-24°C
- **Acceptable**: > 24°C

### 6.2 Air Quality
CO₂ classification:
- **Excellent**: < 600 ppm
- **Good**: 600-800 ppm
- **Fair**: 800-1000 ppm
- **Poor**: > 1000 ppm

### 6.3 Anomaly Detection
- CO₂ > 1000 ppm
- Temperature < 15°C or > 30°C
- Humidity < 20% or > 80%

In [ ]:
silver_stream = parsed_stream \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("processing_time", current_timestamp()) \
    .withColumn(
        "comfort_index",
        when(col("type") == "temperature", 
             when(col("value") < 18, "cold")
             .when((col("value") >= 18) & (col("value") <= 24), "comfortable")
             .otherwise("acceptable")
        ).otherwise("N/A")
    ) \
    .withColumn(
        "air_quality",
        when(col("type") == "co2",
             when(col("value") < 600, "excellent")
             .when((col("value") >= 600) & (col("value") <= 800), "good")
             .when((col("value") > 800) & (col("value") <= 1000), "fair")
             .otherwise("poor")
        ).otherwise("N/A")
    ) \
    .withColumn(
        "anomaly_detected",
        when((col("type") == "co2") & (col("value") > 1000), True)
        .when((col("type") == "temperature") & ((col("value") < 15) | (col("value") > 30)), True)
        .when((col("type") == "humidity") & ((col("value") < 20) | (col("value") > 80)), True)
        .otherwise(False)
    ) \
    .withColumn(
        "data_quality_flag",
        when(
            col("device_id").isNotNull() & 
            col("building").isNotNull() & 
            col("value").isNotNull(),
            "complete"
        ).otherwise("incomplete")
    ) \
    .select(
        col("device_id"),
        col("building"),
        col("floor"),
        col("type"),
        col("value"),
        col("unit"),
        col("event_timestamp"),
        col("anomaly_detected"),
        col("comfort_index"),
        col("air_quality"),
        col("data_quality_flag"),
        col("kafka_partition"),
        col("kafka_offset"),
        col("kafka_timestamp"),
        col("processing_time")
    )

print("✓ Silver transformations configured")

✓ Transformations Silver configurées


## 7. Write to Delta Lake Silver

Streaming configuration to Delta Lake:
- **Format**: Delta Lake (with enrichments)
- **Mode**: Append
- **Checkpoint**: Exactly-once guarantee
- **Trigger**: Process every 10 seconds

In [ ]:
query = silver_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="10 seconds") \
    .start(OUTPUT_PATH)

print("✓ Streaming query started")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")

✓ Streaming query démarré
Query ID: 0caaa3f0-cfe7-4b3f-bdfe-e850ede5f1c0
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


25/12/16 12:18:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


25/12/16 12:18:33 ERROR MicroBatchExecution: Query [id = 0caaa3f0-cfe7-4b3f-bdfe-e850ede5f1c0, runId = 5d4abbf2-2f8e-4cdd-85c2-78dd2d7d5aae] terminated with error
org.apache.spark.sql.delta.DeltaAnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: e1aab3a4-669e-4939-86e7-36552db413ca).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- device_id: string (nullable = true)
-- building: string (nullable = true)
-- floor: integer (nullable = true)
-- type: string (nullable = true)
-- value: double (nullable = true)
-- unit: string (nullable = true)
-- event_timestamp: timestamp (nullable = true)
-- anomaly_detected: boolean (nullable = true)
-- comfort_index: string (

## 8. Streaming Monitoring

Monitoring Kafka → Delta metrics.

In [ ]:
import time

# Wait for processing
time.sleep(20)

print("📊 Streaming status:")
print(f"Is Active: {query.isActive}")
print(f"Recent Progress: {len(query.recentProgress)} batches")

if query.recentProgress:
    latest = query.recentProgress[-1]
    print("\nLatest batch:")
    print(f"  - Batch ID: {latest.get('batchId', 'N/A')}")
    print(f"  - Input Rows: {latest.get('numInputRows', 0)}")
    print(f"  - Process Rate: {latest.get('processedRowsPerSecond', 0):.2f} rows/sec")
    
    # Kafka metrics
    sources = latest.get('sources', [])
    if sources:
        kafka_metrics = sources[0]
        print("\nKafka metrics:")
        print(f"  - Start Offset: {kafka_metrics.get('startOffset', 'N/A')}")
        print(f"  - End Offset: {kafka_metrics.get('endOffset', 'N/A')}")

📊 Statut du streaming:
Is Active: False
Recent Progress: 0 batches


## 9. Silver Data Verification

Batch read for enrichments validation.

In [ ]:
# Read Silver data
silver_df = spark.read.format("delta").load(OUTPUT_PATH)

print(f"📊 Total Silver records: {silver_df.count()}")
print("\n📋 Enriched data preview:")
silver_df.show(10, truncate=False)

25/12/16 12:19:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


📊 Total enregistrements Silver: 3500

📋 Aperçu des données enrichies:
+-----------------+--------+-----+------------------+------+----+-------------------+----------------+-------------+-----------+-----------------+---------------+------------+-----------------------+-----------------------+
|device_id        |building|floor|type              |value |unit|event_timestamp    |anomaly_detected|comfort_index|air_quality|data_quality_flag|kafka_partition|kafka_offset|kafka_timestamp        |processing_time        |
+-----------------+--------+-----+------------------+------+----+-------------------+----------------+-------------+-----------+-----------------+---------------+------------+-----------------------+-----------------------+
|sensor-temp-003  |B       |2    |temperature       |27.5  |°C  |2025-01-12 09:31:43|false           |N/A          |N/A        |complete         |0              |1800        |2025-12-16 11:41:18.594|2025-12-16 11:47:10.003|
|sensor-hum-002   |A       |3    |

In [ ]:
# Comfort Index distribution (temperatures)
print("🌡️  Comfort Index distribution:")
silver_df.filter(col("type") == "temperature") \
    .groupBy("comfort_index").count() \
    .orderBy("comfort_index") \
    .show()

🌡️  Distribution Comfort Index:
+-------------+-----+
|comfort_index|count|
+-------------+-----+
|          N/A|  385|
|   acceptable|  490|
|  comfortable|  630|
+-------------+-----+

+-------------+-----+
|comfort_index|count|
+-------------+-----+
|          N/A|  385|
|   acceptable|  490|
|  comfortable|  630|
+-------------+-----+



In [ ]:
# Air Quality distribution (CO₂)
print("💨 Air Quality distribution:")
silver_df.filter(col("type") == "co2") \
    .groupBy("air_quality").count() \
    .orderBy("air_quality") \
    .show()

💨 Distribution Air Quality:
+-----------+-----+
|air_quality|count|
+-----------+-----+
|  excellent|  189|
|       fair|  161|
|       good|  203|
|       poor|  231|
+-----------+-----+

+-----------+-----+
|air_quality|count|
+-----------+-----+
|  excellent|  189|
|       fair|  161|
|       good|  203|
|       poor|  231|
+-----------+-----+



In [ ]:
# Anomalies by building and type
print("⚠️  Detected anomalies:")
anomalies = silver_df.filter(col("anomaly_detected") == True)
print(f"Total anomalies: {anomalies.count()}")
anomalies.groupBy("building", "type").count() \
    .orderBy(desc("count")) \
    .show()

⚠️  Anomalies détectées:
Total anomalies: 231
Total anomalies: 231
+--------+----+-----+
|building|type|count|
+--------+----+-----+
|       B| co2|  133|
|       A| co2|   98|
+--------+----+-----+

+--------+----+-----+
|building|type|count|
+--------+----+-----+
|       B| co2|  133|
|       A| co2|   98|
+--------+----+-----+



In [ ]:
# Enriched statistics by building
print("📈 Enriched statistics by building:")
silver_df.groupBy("building").agg(
    count("*").alias("total_records"),  # type: ignore[attr-defined]
    countDistinct("device_id").alias("unique_devices"),
    sum(when(col("anomaly_detected") == True, 1).otherwise(0)).alias("anomalies")
).show()

📈 Statistiques enrichies par bâtiment:
+--------+-------------+--------------+---------+
|building|total_records|unique_devices|anomalies|
+--------+-------------+--------------+---------+
|       B|         1792|             5|      133|
|       A|         1708|             5|       98|
+--------+-------------+--------------+---------+

+--------+-------------+--------------+---------+
|building|total_records|unique_devices|anomalies|
+--------+-------------+--------------+---------+
|       B|         1792|             5|      133|
|       A|         1708|             5|       98|
+--------+-------------+--------------+---------+



## 10. Temporal Analysis

Processing latency verification.

In [ ]:
# Calculate processing latency
latency_df = silver_df.withColumn(
    "processing_latency_seconds",
    (unix_timestamp(col("processing_time")) - unix_timestamp(col("kafka_timestamp")))
)

print("⏱️  Latency statistics:")
latency_df.select(
    avg("processing_latency_seconds").alias("avg_latency"),
    min("processing_latency_seconds").alias("min_latency"),
    max("processing_latency_seconds").alias("max_latency")
).show()

⏱️  Statistiques de latence:
+------------------+-----------+-----------+
|       avg_latency|min_latency|max_latency|
+------------------+-----------+-----------+
|338.93228571428574|          5|        483|
+------------------+-----------+-----------+

+------------------+-----------+-----------+
|       avg_latency|min_latency|max_latency|
+------------------+-----------+-----------+
|338.93228571428574|          5|        483|
+------------------+-----------+-----------+



## 11. Kafka Metadata

Kafka message tracing verification.

In [ ]:
print("📡 Distribution by Kafka partition:")
silver_df.groupBy("kafka_partition").count() \
    .orderBy("kafka_partition") \
    .show()

📡 Distribution par partition Kafka:
+---------------+-----+
|kafka_partition|count|
+---------------+-----+
|              0| 3500|
+---------------+-----+

+---------------+-----+
|kafka_partition|count|
+---------------+-----+
|              0| 3500|
+---------------+-----+



In [ ]:
print("📡 Kafka offsets (min/max):")
silver_df.select(
    min("kafka_offset").alias("min_offset"),
    max("kafka_offset").alias("max_offset")
).show()

📡 Offsets Kafka (min/max):
+----------+----------+
|min_offset|max_offset|
+----------+----------+
|         0|      3499|
+----------+----------+

+----------+----------+
|min_offset|max_offset|
+----------+----------+
|         0|      3499|
+----------+----------+



## 12. Delta Lake History

Silver transaction audit.

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, OUTPUT_PATH)
print("📜 Silver transaction history:")
delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(10, truncate=False)

📜 Historique des transactions Silver:
+-------+-----------------------+----------------+----------------------------------------------------------------------------------------+
|version|timestamp              |operation       |operationMetrics                                                                        |
+-------+-----------------------+----------------+----------------------------------------------------------------------------------------+
|34     |2025-12-16 11:48:32.271|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 100, numOutputBytes -> 7240, numAddedFiles -> 1}|
|33     |2025-12-16 11:48:25.793|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 100, numOutputBytes -> 7237, numAddedFiles -> 1}|
|32     |2025-12-16 11:48:20.681|STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 100, numOutputBytes -> 7327, numAddedFiles -> 1}|
|31     |2025-12-16 11:48:15.92 |STREAMING UPDATE|{numRemovedFiles -> 0, numOutputRows -> 100, numOutputBytes -> 7253, num

## 13. Stop Streaming

In [ ]:
# Stop streaming query
query.stop()
time.sleep(2)
print(f"✓ Streaming stopped (Active: {query.isActive})")

✓ Streaming arrêté (Active: False)


## Key Concepts Illustrated

### 1. Kafka Integration
- Real-time message consumption
- JSON deserialization
- Kafka metadata (partition, offset, timestamp)

### 2. Silver Transformations
- **Business Logic**: Comfort Index, Air Quality
- **Enrichment**: Adding calculated columns
- **Quality**: Anomaly detection

### 3. Exactly-Once Semantics
- Kafka offsets + Delta checkpoints
- No-duplication guarantee
- Operations idempotence

### 4. Monitoring
- Processing metrics (rows/sec)
- End-to-end latency
- Data distribution

### 5. Medallion Architecture
- **Silver**: Cleaned and enriched data
- Ready for Gold layer (aggregations)

### 6. Delta Lake (Silver)
- Schema evolution
- Time travel for audit
- ACID transactions maintained